# NB8 — Feature Engineering và leakage
Đo feature nhân quả, target encoding, PIT join và on-demand feature.
Dữ liệu tổng hợp dùng seed cố định. Kết quả lưu ở
`submission/results/nb8_feature_engineering.json`.
ODFV dùng Feast project riêng `lab19_odfv` với SQLite/file, kể cả khi NB4
dùng Redis/Postgres. Không thay registry hoặc ba feature view của NB4.

In [1]:
import _setup  # noqa: F401
import os
import subprocess
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from _advanced import Evidence
from app.features import (auc, generate_events, latest_join, leakage_experiment,
                          leaked_row_fraction, pit_join, window_aggregates)

ROOT = Path(_setup.__file__).resolve().parent.parent
evidence = Evidence("nb8_feature_engineering.json", notebook="NB8", seed=42)

## 1. Event log và feature theo cửa sổ
Bốn họ đầu: aggregation, ratio, lag/delta và recency. Mỗi feature chỉ dùng
lịch sử trước sự kiện. Dòng đầu mỗi user không có lịch sử nên lag/ratio là NaN.

In [2]:
events = generate_events(n_users=200, n_days=30, seed=42)
print(f"Events={len(events)}; users={events.user_id.nunique()}; "
      f"sessions={events.session_id.nunique()}; click rate={events.clicked.mean():.3f}")
print(events.head(3).to_string(index=False))
features = window_aggregates(events)
columns = ["searches_1h", "searches_24h", "searches_7d", "query_len_vs_user_avg",
           "prev_query_len", "query_len_delta", "seconds_since_last"]
print(features[columns].describe().loc[["mean", "50%", "max"]].round(2).to_string())

Events=9019; users=200; sessions=8141; click rate=0.569
user_id                     event_timestamp      topic session_id  query_len  results_shown  clicked
  u_048 2026-07-01 00:00:34.132897690+00:00 networking s_048_0000         10             15        0
  u_080 2026-07-01 00:02:11.225255095+00:00   data_eng s_080_0000         23             12        0
  u_093 2026-07-01 00:04:41.999966662+00:00      cloud s_093_0000         16             13        1
      searches_1h  searches_24h  searches_7d  query_len_vs_user_avg  prev_query_len  query_len_delta  seconds_since_last
mean         0.07          1.69        10.66                   1.03           13.64            -0.01            55695.74
50%          0.00          1.00        11.00                   1.01           14.00             0.00            34931.18
max          2.00         10.00        31.00                   8.00           24.00            21.00           888567.46


## 2. AUC train/holdout của feature nhân quả
Dùng AUC trực tiếp của feature, không train classifier. Đây là minh họa IID;
bài toán dự báo tương lai còn cần chia theo thời gian/user tùy cách triển khai.

In [3]:
mask = np.random.default_rng(0).random(len(features)) < 0.7
honest = []
for column in ("searches_24h", "searches_7d", "seconds_since_last"):
    train_auc = auc(features.loc[mask, column], features.loc[mask, "clicked"])
    test_auc = auc(features.loc[~mask, column], features.loc[~mask, "clicked"])
    honest.append({"feature": column, "train_auc": train_auc, "holdout_auc": test_auc,
                   "gap": train_auc - test_auc})
print(pd.DataFrame(honest).round(3).to_string(index=False))

           feature  train_auc  holdout_auc    gap
      searches_24h      0.592        0.580  0.012
       searches_7d      0.646        0.666 -0.020
seconds_since_last      0.435        0.435  0.000


## 3. Họ categorical: target encoding leakage
Chia train/holdout trước, fit encoder trên train. `target-naive` dùng cả nhãn
của chính dòng train; `target-in-fold` mã hóa dòng đó từ các fold còn lại.
Mục tiêu: gap naive >0.30 với `session_id`, gap in-fold gần 0.

In [4]:
session_leakage = leakage_experiment(events, "session_id")
user_leakage = leakage_experiment(events, "user_id")
print("session_id:\n", session_leakage.round(3).to_string(index=False))
print("\nuser_id:\n", user_leakage.round(3).to_string(index=False))
session_scores = session_leakage.set_index("encoding")
print("Naive gap:", session_scores.loc["target-naive", "gap"])
print("In-fold gap:", session_scores.loc["target-in-fold", "gap"])

session_id:
       encoding  train_auc  test_auc    gap
     frequency      0.521     0.516  0.005
  target-naive      0.999     0.522  0.477
target-in-fold      0.511     0.522 -0.011

user_id:
       encoding  train_auc  test_auc   gap
     frequency      0.726     0.708 0.018
  target-naive      0.749     0.698 0.052
target-in-fold      0.714     0.698 0.016
Naive gap: 0.4768965540039024
In-fold gap: -0.011065958029036094


## 4. Latest join so với point-in-time join
Feature snapshot ghi số hoạt động tích lũy. Latest join kéo snapshot tương lai
về dòng huấn luyện; PIT lấy snapshot gần nhất có timestamp <= thời điểm sự kiện.
Đo tỷ lệ dòng bị rò và chênh lệch AUC, không gán trước dấu của chênh lệch.

In [5]:
snapshots = events[["user_id", "event_timestamp"]].copy().sort_values("event_timestamp")
snapshots["feature_value"] = snapshots.groupby("user_id").cumcount() + 1
sample = np.random.default_rng(1).random(len(events)) < 0.4
entities = events.loc[sample, ["user_id", "event_timestamp", "clicked"]].copy()
latest, pit = latest_join(entities, snapshots), pit_join(entities, snapshots)
pit_result = {
    "rows": len(entities), "leaked_row_fraction": leaked_row_fraction(entities, snapshots),
    "latest_auc": auc(latest["feature_value"], latest["clicked"]),
    "pit_auc": auc(pit["feature_value"], pit["clicked"]),
}
pit_result["auc_difference"] = pit_result["latest_auc"] - pit_result["pit_auc"]
print(f"Dòng bị rò: {pit_result['leaked_row_fraction']:.2%} / {len(entities)} rows")
print(f"Latest AUC: {pit_result['latest_auc']:.3f}; PIT AUC: {pit_result['pit_auc']:.3f}")
print(f"Delta latest - PIT: {pit_result['auc_difference']:+.3f}")

Dòng bị rò: 98.19% / 3535 rows
Latest AUC: 0.715; PIT AUC: 0.595
Delta latest - PIT: +0.120


## 5. Họ embedding: vector làm feature
Dùng vector profile của user từ trung bình ba vector tài liệu đã đọc để minh họa.
Trong bài này dùng vector nhỏ giả lập, không tải thêm model. Hệ thật có thể dùng
vector NB1 và chỉ tổng hợp lịch sử **trước** thời điểm request để tránh leakage.

In [6]:
history_vectors = np.random.default_rng(42).normal(size=(3, 8))
user_vector = history_vectors.mean(axis=0)
user_vector /= np.linalg.norm(user_vector)
print("Ví dụ user embedding (synthetic, 8d):", user_vector.round(3))
print("Đây là ví dụ họ feature thứ 6; không dùng vector giả này để chấm AUC.")

Ví dụ user embedding (synthetic, 8d): [ 0.125 -0.542  0.476  0.317 -0.393 -0.163  0.345 -0.253]
Đây là ví dụ họ feature thứ 6; không dùng vector giả này để chấm AUC.


## 6. On-demand feature view
`amount_vs_avg = amount / avg_amount_7d`: baseline lưu trong Feast, `amount`
đến từ request. Chạy bằng Python của kernel hiện tại để không nhầm môi trường.
Nạp lại cửa sổ 7 ngày tới thời điểm hiện tại, nên chạy lại vẫn lấy dữ liệu mới.
Log stdout/stderr được in đầy đủ nếu Feast gặp lỗi.

In [7]:
repo = ROOT / "app/feast_repo_ondemand"
command_logs = []


def run_command(args, cwd=ROOT):
    process = subprocess.run(
        [str(arg) for arg in args], cwd=cwd, capture_output=True, text=True,
        encoding="utf-8", errors="replace",
        env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        timeout=180,
        creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
    )
    print(process.stdout)
    if process.stderr:
        print(process.stderr)
    command_logs.append({"args": [str(arg) for arg in args], "exit_code": process.returncode,
                         "stdout": process.stdout, "stderr": process.stderr})
    process.check_returncode()


run_command([sys.executable, ROOT / "scripts/gen_spend.py"])
feast_executable = Path(sys.executable).with_name("feast.exe" if os.name == "nt" else "feast")
assert feast_executable.exists(), "Chọn kernel Python (Lab 19) có cài Feast"
feast_cli = [str(feast_executable)]
run_command([*feast_cli, "apply"], cwd=repo)
end = datetime.now(timezone.utc)
start = end - timedelta(days=7)
run_command([*feast_cli, "materialize", start.isoformat(), end.isoformat()], cwd=repo)

wrote D:\AI_Thucchien\K4-Track2-Day19-VectorFeatureStore-Lab\app\feast_repo_ondemand\data\user_spend.parquet (150 rows, 50 users)

No project found in the repository. Using project name lab19_odfv defined in feature_store.yaml
Applying changes for project lab19_odfv
Created project lab19_odfv
Created entity user
Created feature view user_spend_stats
Created on demand feature view amount_vs_avg

Created sqlite table lab19_odfv_user_spend_stats


D:\AI_Thucchien\K4-Track2-Day19-VectorFeatureStore-Lab\app\feast_repo_ondemand\definitions.py:28: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'user'.
  user = Entity(name="user", join_keys=["user_id"])
D:\AI_Thucchien\K4-Track2-Day19-VectorFeatureStore-Lab\.venv\lib\site-packages\feast\feature_store.py:1165: RuntimeWarning: On demand feature view is an experimental feature. This API is stable, but the functionality does not scale well for offline retrieval
  warnings.warn(



In [8]:
from feast import FeatureStore

store = FeatureStore(repo_path=str(repo))
requests = [
    {"user_id": "u_000", "amount": 100_000.0},
    {"user_id": "u_000", "amount": 15_000_000.0},
    {"user_id": "u_001", "amount": 250_000.0},
]
online = store.get_online_features(
    features=["user_spend_stats:avg_amount_7d", "amount_vs_avg:amount_vs_avg",
              "amount_vs_avg:is_spike"], entity_rows=requests,
).to_dict()
odfv_rows = []
for i, request in enumerate(requests):
    average = online["avg_amount_7d"][i]
    ratio = online["amount_vs_avg"][i]
    assert average is not None and average > 0, "Baseline chưa được materialize"
    assert ratio is not None, "ODFV chưa trả về amount_vs_avg"
    odfv_rows.append({**request, "avg_amount_7d": average,
                      "amount_vs_avg": ratio, "is_spike": online["is_spike"][i]})
print(pd.DataFrame(odfv_rows).round(3).to_string(index=False))

user_id     amount  avg_amount_7d  amount_vs_avg  is_spike
  u_000   100000.0     3566076.21          0.028         0
  u_000 15000000.0     3566076.21          4.206         1
  u_001   250000.0     3718992.16          0.067         0


## 7. Lưu bằng chứng
Chụp mục 3, 4, 6 và Ctrl+S. Ngưỡng `abs(in-fold gap)<0.10` là mức kiểm tra
“gần 0” đã dùng trong tests của lab. Hai request cùng user phải dùng cùng
baseline nhưng ra hai ratio khác nhau, đúng công thức.

In [9]:
evidence.finish(
    checks={
        "session_naive_gap_above_030": session_scores.loc["target-naive", "gap"] > 0.30,
        "session_infold_gap_near_zero": abs(session_scores.loc["target-in-fold", "gap"]) < 0.10,
        "latest_join_has_future_rows": pit_result["leaked_row_fraction"] > 0,
        "pit_auc_measured": all(np.isfinite(pit_result[key])
                                for key in ("latest_auc", "pit_auc", "auc_difference")),
        "same_user_same_stored_average": odfv_rows[0]["avg_amount_7d"] == odfv_rows[1]["avg_amount_7d"],
        "same_user_different_ratios": odfv_rows[0]["amount_vs_avg"] != odfv_rows[1]["amount_vs_avg"],
        "odfv_matches_formula": all(np.isclose(row["amount_vs_avg"], row["amount"] / row["avg_amount_7d"])
                                    for row in odfv_rows),
    },
    n_events=len(events), honest_features=honest,
    session_leakage=session_leakage.to_dict("records"),
    user_leakage=user_leakage.to_dict("records"), pit_join=pit_result,
    on_demand=odfv_rows, feast_project=store.project, commands=command_logs,
)

PASS: session_naive_gap_above_030
PASS: session_infold_gap_near_zero
PASS: latest_join_has_future_rows
PASS: pit_auc_measured
PASS: same_user_same_stored_average
PASS: same_user_different_ratios
PASS: odfv_matches_formula
Saved: D:\AI_Thucchien\K4-Track2-Day19-VectorFeatureStore-Lab\submission\results\nb8_feature_engineering.json (status=passed)
